## Fase 2: ETL - Integración y Limpieza del Maestro de Socios
Aquí se unifican las bases de datos separadas de socios activos y socios que se han dado de baja. Se analiza estadísticamente la proporción de cada grupo para comprender el balance de clases (identificando un churn rate histórico del 41.8% frente a un 58.1% de usuarios activos) y se guarda el dataset maestro consolidado en formato CSV, dejándolo listo para cruzarlo con el comportamiento de visitas.

In [ ]:
import pandas as pd

# 1. Definimos las rutas relativas para localizar los archivos fuente de socios activos y bajas
ruta_activos = "../../datos/socios_activos.xlsx"
ruta_bajas = "../../datos/socios_bajas.xlsx"

# Cargamos los datos de socios activos y bajas desde los archivos Excel
df_activos = pd.read_excel(ruta_activos)
df_bajas = pd.read_excel(ruta_bajas)

# 2. Creacion de la Variable Objetivo (Target) para el modelo predictivo
df_activos['es_baja'] = 0  # 0 = Socio activo (No ha abandonado)
df_bajas['es_baja'] = 1    # 1 = Socio inactivo (Dado de baja)

# 3. Concatenamos ambos datasets verticalmente en una unica tabla global
df_socios = pd.concat([df_activos, df_bajas], ignore_index=True)

# 4. Filtramos el dataset para analizar unicamente a los socios con cuota mensual
df_socios = df_socios[df_socios['tipo_cuota'] == 'MENSUAL']

# 5. Eliminamos variables irrelevantes o redundantes para evitar ruido en el modelo
# Quitamos columnas textuales o financieras que no aportan valor predictivo directo
columnas_basura = ['saldo', 'estado_socio'] 
df_socios = df_socios.drop(columns=columnas_basura, errors='ignore')

# 6. Comprobacion de la estructura final y balanceo de la variable objetivo
print(f"Total de socios validos para el modelo: {len(df_socios)}")
print(f"\nTotal de socios activos (0): {len(df_socios[df_socios['es_baja'] == 0])}")
print(f"\nTotal de socios dados de baja (1): {len(df_socios[df_socios['es_baja'] == 1])}")
print("\nPorcentaje de Activos (0) vs Bajas (1):")
print(df_socios['es_baja'].value_counts(normalize=True) * 100)

Total de socios validos para el modelo: 8658

Total de socios activos (0): 5038

Total de socios dados de baja (1): 3620

Porcentaje de Activos (0) vs Bajas (1):
es_baja
0    58.188958
1    41.811042
Name: proportion, dtype: float64


## Exportación Final del Maestro de Socios

In [3]:
import os

# 1. Definimos el directorio de destino para los datos procesados
ruta_guardado = "../../datos/procesados"

# 2. Creamos la carpeta en caso de que no exista previamente
os.makedirs(ruta_guardado, exist_ok=True)

# 3. Exportamos el DataFrame a un archivo CSV omitiendo el indice
ruta_archivo_final = f"{ruta_guardado}/socios_historico_total.csv"
df_socios.to_csv(ruta_archivo_final, index=False)

# 4. Imprimimos un mensaje de confirmacion con la ruta exacta
print(f"\nArchivo procesado y guardado fisicamente con exito en:\n{ruta_archivo_final}")


Archivo procesado y guardado fisicamente con exito en:
../../datos/procesados/socios_historico_total.csv
